## Merge Author Profiles — Coauthor-Only One-ORCID-Anchor, Size 2-9, No-Middle

**Phase 4b/c** (combined) — coauthor-only extension of Phase 4. Same Phase-4 recipe (no-middle parsed names, exactly one ORCID-bearing anchor, all per-profile guards) but inverts the per-pair signal: `inst_overlap = FALSE AND n_shared_coauthors >= 3`. Picks up splinters where coauthor signal is strong but no institution overlap exists. Combines the size-2 (Phase 4b) and size-3-9 (Phase 4c) cohorts into a single notebook — the recipe is the same; only the group-size guard and the per-pair filtering semantics differ.

Companion to:
- `MergeAuthorsNoMiddleAnchor` (Phase 4) — same scope, but `inst_overlap = TRUE AND COA >= 2`
- `MergeAuthorsRichName` (Phase 2) / `MergeAuthorsOrcidAnchorMulti` (Phase 3b) / `MergeAuthorsRichNameMulti` (Phase 3c) — rich-name (middle ≥ 3 char word required)

**Why this rule next:** Phase 4 covered the no-middle 1-ORCID-anchor cohort *with* inst overlap. The residual class is no-INST/COA-only — same person on different works showing different affiliations, where MatchAuthors can't bridge them via inst overlap. Without an INST anchor, the precision risk is career-move ambiguity (same parsed name, distinct careers, accidental coauthor bridge); we compensate with a higher COA floor (3, vs Phase 4's 2) and the same Phase-4 character-class guards. Extending to size 3-9 picks up the **Priem A5108365034 case** and similar multi-fragment ORCID-anchor groups.

**Per-pair semantics for size 3-9:** for each `(anchor, straggler_i)` pair in a group, the COA filter is applied independently. A group with 1 ORCID + 3 stragglers where 2 stragglers pass and 1 fails will emit only the 2 passing pairs as merge candidates. The failing straggler stays unmerged.

**Audit progression (2026-05-03, 6 rounds across both portions):**

| Cohort | Round | n | Result |
|---|---|---|---|
| Size 2 (Phase 4b) | v1 (seed=2026) | 50 | 50/50 = 100% |
| Size 2 (Phase 4b) | v2 (seed=3037) | 50 | 50/50 = 100% |
| Size 2 (Phase 4b) | v3 (seed=4148) | 100 | 99/100 = 99% |
| **Size 2 subtotal** | | **200** | **199/200 = 99.5%** |
| Size 3-9 (Phase 4c) | v1 (seed=5259+) | 50 | 49/50 = 98% |
| Size 3-9 (Phase 4c) | v2 (seed=6370+) | 50 | 49/50 = 98% |
| Size 3-9 (Phase 4c) | v3 (seed=7481+) | 100 | 97/100 = 97% (realistic), 94/100 = 94% (conservative) |
| **Size 3-9 subtotal** | | **200** | **195-196/200 ≈ 98% realistic** |
| **Combined** | | **400** | **~98% weighted realistic** |

**Volume estimate:**

| Group size | Pairs |
|---|---|
| 2 | 33,913 |
| 3 | 30,764 |
| 4 | 25,952 |
| 5 | 21,064 |
| 6 | 16,547 |
| 7 | 12,942 |
| 8 | 10,272 |
| 9 | 8,075 |
| **Total** | **~159,529 pairs / loser profiles** |

**Precision guards (identical to Phase 4 except group-size cap and final signal):**
1. First/last length ≥ 2 AFTER stripping trailing period.
2. No-middle filter (no word ≥ 3 chars in `p_middle`).
3. First-name vowel guard.
4. Dominant raw_name majority (≥ 50%).
5. Year-in-full_name guard.
6. Middle-initial-from-dom_raw guard.
7. Full_name implied-initial agreement guard (`fn_initials_ok`).
8. Full_name self-consistency (`p_last` whole token in full_name).
9. **Group size 2-9** (was = 2 in Phase 4 / Phase 4b-only).
10. Sink cap (`works_count <= 5,000`).
11. Exactly one ORCID-bearing profile per group.
12. **Per-pair signal:** `inst_overlap = FALSE AND n_shared_coauthors >= 3` (applied per pair, not per group).

**Winner selection:** ORCID-bearing anchor wins, by construction. (Same as Phase 3b / Phase 4 — works_count tiebreakers don't apply because the ORCID profile always wins.)

**Loser handling:** *No DELETE.* Same MERGE-then-soft-zero-via-CreateAuthors approach as Phases 1 / 2 / 3b / 3c / 4.

**Rollback plan:** No automatic rollback. Audit log `openalex.authors.author_merge_log_coauthor_anchor` captures the anchor's ORCID, parsed name, signal flags, and per-profile metadata; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.

**Known residual failure modes (out of scope):**
- **Multi-overmerge straggler bridged by misattribution:** the dominant FP class. Straggler is itself a 2-5-way overmerge whose 1 misattributed paper creates a coauthor bridge to the anchor. Examples from audit: Lawrence (educator + radiology chemist), Knowles (5-way Knowles overmerge), Vargas (3-way Vargas overmerge), Jane Donovan (Drexel nurse + WVU historian via 1 cervix-cancer paper). ~1-2% of pairs.
- **Different-careers same-parsed-name:** Tsunoda v3 case (Hokkaido forestry vs Tokyo radiology) — two distinct humans with common Japanese name, COA bridge via shared common-name coauthors. Irreducible without external data.
- **Comma-form parser inversion on dominant raw:** the Priem A5131149981 case — dominant raw `"Jason, Priem"` parses to `('priem', '', 'jason')` while canonical's parses to `('jason', '', 'priem')`. Different parsed keys → never grouped. Needs a CreateAuthorNames parser fix or unordered first/last grouping.
- **Anchor pre-existing overmerges:** ~3% of audited anchors are themselves overmerges (Almir Sales / Pascal Ravassard / Talia Shepherd / Yi-Kang Pu classes). Phase 4b/c doesn't worsen them — straggler matches the anchor's dominant persona — but anchor splits remain a separate concern.

### Cell 1: Build merge candidates table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_coauthor_anchor AS
WITH
all_profiles AS (
  SELECT id AS author_id, orcid, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids
  FROM openalex.authors.openalex_authors
),
ra_counts AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN all_profiles n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >= 50% majority (Phase 2 recipe).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last). Phase 4 filters:
  --  * first/last >= 2 chars AFTER stripping trailing period (initials-first defense)
  --  * middle has NO word >= 3 chars (no-middle cohort -- inverse of rich-name)
  --  * first contains a vowel (rejects 'SJ', 'DJ' all-initials cases)
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '[.]$', '') AS p_first,
         REGEXP_REPLACE(an.parsed_name.middle, '[.]$', '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '[.]$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(REGEXP_REPLACE(an.parsed_name.first, '[.]$', '')) >= 2
    AND LENGTH(REGEXP_REPLACE(an.parsed_name.last,  '[.]$', '')) >= 2
    AND SIZE(FILTER(
          SPLIT(REGEXP_REPLACE(COALESCE(an.parsed_name.middle,''), '[.]', ''), ' +'),
          x -> LENGTH(x) >= 3
        )) = 0
    AND REGEXP_LIKE(LOWER(REGEXP_REPLACE(an.parsed_name.first, '[.]', '')), '[aeiouy]')
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last,
         (n.orcid IS NOT NULL AND n.orcid <> '') AS has_orcid,
         LOWER(SUBSTR(REGEXP_REPLACE(p.p_middle, '[. ]', ''), 1, 1)) AS p_middle_initial,
         ARRAY_DISTINCT(FILTER(
           SPLIT(LOWER(REGEXP_REPLACE(TRANSLATE(COALESCE(n.full_name, ''),
             'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
             'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
             '[^a-zA-Z]', ' ')), ' +'),
           x -> LENGTH(x) = 1 AND x RLIKE '[a-z]'
         )) AS fn_initials
  FROM all_profiles    n
  JOIN parsed_dominant p ON p.author_id = n.author_id
),
profile_consistent AS (
  -- Per-profile guards:
  --  * full_name not null
  --  * year-in-full_name guard (no 4-digit runs -- catches life-dates pollution)
  --  * full_name self-consistency: must contain p_last as a whole token
  --  * middle-initial-from-dom_raw guard: when p_middle is non-empty,
  --    full_name must contain that initial as a single-letter token
  SELECT pwd.*,
    CASE
      WHEN full_name IS NULL THEN FALSE
      WHEN REGEXP_LIKE(full_name, '[0-9]{4}') THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ', p_last, ' ')
      ) = 0 THEN FALSE
      WHEN p_middle_initial <> ''
           AND INSTR(
             CONCAT(' ', LOWER(REGEXP_REPLACE(
               TRANSLATE(full_name,
                 'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
                 'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
               '[^a-zA-Z]', ' ')), ' '),
             CONCAT(' ', p_middle_initial, ' ')
           ) = 0
        THEN FALSE
      ELSE TRUE
    END AS full_name_ok
  FROM profile_with_dom pwd
),
qualifying_groups AS (
  -- Group size 2-9, exactly one ORCID-bearing profile, sink cap, all profiles full_name_ok.
  SELECT p_first, p_middle, p_last
  FROM profile_consistent
  GROUP BY p_first, p_middle, p_last
  HAVING COUNT(*) BETWEEN 2 AND 9
     AND MAX(works_count) <= 5000
     AND COUNT(CASE WHEN has_orcid THEN 1 END) = 1
     AND COUNT(DISTINCT CASE WHEN has_orcid THEN orcid END) = 1
     AND COUNT(*) = COUNT(CASE WHEN full_name_ok THEN 1 END)
),
group_profiles AS (
  SELECT pc.*
  FROM profile_consistent pc
  JOIN qualifying_groups g
    ON pc.p_first = g.p_first AND pc.p_middle = g.p_middle AND pc.p_last = g.p_last
),
anchor AS (
  SELECT p_first, p_middle, p_last,
         author_id AS anchor_id,
         orcid     AS anchor_orcid,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS anchor_inst,
         fn_initials AS anchor_fn_initials
  FROM group_profiles
  WHERE has_orcid
),
straggler AS (
  -- For size 3-9 groups, this CTE produces multiple straggler rows per parsed name,
  -- each later joined to the single anchor of that group via parsed-name equality.
  SELECT p_first, p_middle, p_last,
         author_id AS straggler_id,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS straggler_inst,
         fn_initials AS straggler_fn_initials
  FROM group_profiles
  WHERE NOT has_orcid
),
pair_inst AS (
  -- Compute inst_overlap and the fn_initials agreement guard at the pair level.
  -- For size N groups this produces (N - 1) (anchor, straggler) pair rows.
  SELECT a.p_first, a.p_middle, a.p_last,
         a.anchor_id, s.straggler_id,
         (SIZE(ARRAY_INTERSECT(a.anchor_inst, s.straggler_inst)) > 0) AS inst_overlap,
         (SIZE(a.anchor_fn_initials) = 0
          OR SIZE(s.straggler_fn_initials) = 0
          OR SIZE(ARRAY_INTERSECT(a.anchor_fn_initials, s.straggler_fn_initials)) > 0) AS fn_initials_ok
  FROM anchor a
  JOIN straggler s
    ON a.p_first = s.p_first AND a.p_middle = s.p_middle AND a.p_last = s.p_last
),
no_inst_pairs AS (
  -- Phase 4b/c: keep only pairs WITHOUT inst overlap (residual class Phase 4 missed).
  SELECT * FROM pair_inst WHERE inst_overlap = FALSE AND fn_initials_ok = TRUE
),
group_member_coauthors AS (
  -- Per profile, the set of distinct coauthor author_ids on its work_authors rows.
  SELECT wa1.author_id AS profile_id, wa2.author_id AS coauthor_id
  FROM openalex.works.work_authors wa1
  JOIN openalex.works.work_authors wa2
    ON wa2.work_id = wa1.work_id
   AND wa2.author_id <> wa1.author_id
  WHERE wa1.author_id IN (SELECT anchor_id    FROM no_inst_pairs)
     OR wa1.author_id IN (SELECT straggler_id FROM no_inst_pairs)
  GROUP BY wa1.author_id, wa2.author_id
),
pair_coauthor AS (
  -- Coauthors shared between anchor and straggler, EXCLUDING any other group member
  -- (intra-group coauthorship doesn't count; this is critical for size 3-9 groups
  -- where stragglers have already been merged together by some prior process).
  SELECT p.p_first, p.p_middle, p.p_last,
         p.anchor_id, p.straggler_id,
         COUNT(DISTINCT ac.coauthor_id) AS n_shared_coauthors
  FROM no_inst_pairs p
  JOIN group_member_coauthors ac ON ac.profile_id = p.anchor_id
  JOIN group_member_coauthors sc ON sc.profile_id = p.straggler_id
                                AND sc.coauthor_id = ac.coauthor_id
  WHERE ac.coauthor_id NOT IN (SELECT anchor_id    FROM no_inst_pairs)
    AND ac.coauthor_id NOT IN (SELECT straggler_id FROM no_inst_pairs)
  GROUP BY p.p_first, p.p_middle, p.p_last, p.anchor_id, p.straggler_id
)
SELECT
  ps.p_first, ps.p_middle, ps.p_last,
  ps.anchor_id,
  a.anchor_orcid,
  ps.straggler_id,
  ps.inst_overlap,
  pc.n_shared_coauthors,
  -- Anchor metadata
  ap.full_name      AS anchor_full_name,
  ap.works_count    AS anchor_works_count,
  ap.cited_by_count AS anchor_cited_by_count,
  ap.created_date   AS anchor_created_date,
  ap.block_key      AS anchor_block_key,
  -- Straggler (loser) metadata
  sp.full_name      AS straggler_full_name,
  sp.orcid          AS straggler_orcid,
  sp.works_count    AS straggler_works_count,
  sp.cited_by_count AS straggler_cited_by_count,
  sp.created_date   AS straggler_created_date,
  sp.block_key      AS straggler_block_key
FROM no_inst_pairs ps
JOIN pair_coauthor pc
  ON pc.anchor_id = ps.anchor_id AND pc.straggler_id = ps.straggler_id
JOIN anchor a
  ON a.p_first = ps.p_first AND a.p_middle = ps.p_middle
 AND a.p_last  = ps.p_last  AND a.anchor_id = ps.anchor_id
JOIN group_profiles ap ON ap.author_id = ps.anchor_id
JOIN group_profiles sp ON sp.author_id = ps.straggler_id
-- Phase 4b/c signal floor: REQUIRE inst_overlap = FALSE AND n_shared_coauthors >= 3.
-- Applied per pair, so a size-3+ group with 1 passing + 1 failing straggler emits
-- only the passing straggler's pair as a merge candidate.
WHERE pc.n_shared_coauthors >= 3

### Cell 2: Validation statistics

In [ ]:
%sql
WITH group_size_per_anchor AS (
  SELECT anchor_id, COUNT(DISTINCT straggler_id) + 1 AS group_size_emitted
  FROM openalex.authors.merge_candidates_coauthor_anchor
  GROUP BY anchor_id
)
SELECT
  COUNT(DISTINCT c.anchor_id)                                                AS n_anchors,
  COUNT(*)                                                                   AS n_losers,
  COUNT(DISTINCT c.anchor_id, c.straggler_id)                                AS n_pairs,
  SUM(c.straggler_works_count)                                               AS works_to_rewrite,
  COUNT(CASE WHEN g.group_size_emitted = 2 THEN 1 END)                       AS pairs_in_emitted_size_2,
  COUNT(CASE WHEN g.group_size_emitted = 3 THEN 1 END)                       AS pairs_in_emitted_size_3,
  COUNT(CASE WHEN g.group_size_emitted >= 4 THEN 1 END)                      AS pairs_in_emitted_size_4plus,
  COUNT(CASE WHEN c.n_shared_coauthors >=  5 THEN 1 END)                     AS pairs_coa_5plus,
  COUNT(CASE WHEN c.n_shared_coauthors >= 10 THEN 1 END)                     AS pairs_coa_10plus,
  COUNT(CASE WHEN c.n_shared_coauthors >= 25 THEN 1 END)                     AS pairs_coa_25plus,
  COUNT(CASE WHEN c.straggler_orcid IS NOT NULL AND c.straggler_orcid <> ''
              THEN 1 END)                                                    AS losers_with_orcid_unexpected,
  PERCENTILE_APPROX(c.n_shared_coauthors, ARRAY(0.5, 0.95, 0.99))            AS coauthor_pctiles,
  PERCENTILE_APPROX(c.straggler_works_count, ARRAY(0.5, 0.95, 0.99))         AS straggler_works_pctiles
FROM openalex.authors.merge_candidates_coauthor_anchor c
JOIN group_size_per_anchor g ON g.anchor_id = c.anchor_id

### Cell 3: Spot-check sample (manual review)

In [ ]:
%sql
WITH sampled_anchors AS (
  SELECT DISTINCT anchor_id
  FROM openalex.authors.merge_candidates_coauthor_anchor
  ORDER BY RAND() LIMIT 100
)
SELECT c.anchor_id, c.anchor_orcid, c.p_first, c.p_middle, c.p_last,
       c.anchor_full_name, c.anchor_works_count, c.anchor_cited_by_count,
       c.straggler_id, c.straggler_full_name,
       c.straggler_works_count, c.straggler_cited_by_count,
       c.inst_overlap, c.n_shared_coauthors
FROM openalex.authors.merge_candidates_coauthor_anchor c
JOIN sampled_anchors s ON s.anchor_id = c.anchor_id
ORDER BY c.anchor_id, c.straggler_works_count DESC

### Cell 4: Create audit log

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_coauthor_anchor AS
SELECT
  anchor_id              AS winner_author_id,
  anchor_orcid           AS winner_orcid,
  straggler_id           AS loser_author_id,
  straggler_orcid        AS loser_orcid,
  straggler_full_name    AS loser_full_name,
  straggler_works_count  AS loser_works_count,
  straggler_cited_by_count AS loser_cited_by_count,
  straggler_created_date AS loser_created_date,
  p_first, p_middle, p_last,
  inst_overlap,
  n_shared_coauthors,
  current_timestamp()    AS merge_timestamp
FROM openalex.authors.merge_candidates_coauthor_anchor

### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_coauthor_anchor AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_coauthor_anchor log
  ON wa.author_id = log.loser_author_id

### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners

*No DELETE step. Phases 1 / 2 / 3b / 3c / 4 all use the MERGE-then-soft-zero-via-CreateAuthors approach; this notebook follows that pattern.*

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_coauthor_anchor
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()

### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_coauthor_anchor

### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors

In [ ]:
%sql
WITH log AS (
  SELECT loser_author_id, loser_works_count
  FROM openalex.authors.author_merge_log_coauthor_anchor
),
loser_state AS (
  SELECT l.loser_author_id, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles,
  MAX(current_works_count)                                       AS max_remaining_works
FROM loser_state